# 5단계 · 원래 단위로 되돌리기와 '과거 기준선' 이탈도  [외부 데이터 사용 — B층]

**하는 일**
1. 공식 파일의 표준화값을 원래 단위(초, mm, bar, ℃)로 되돌립니다.
2. 비라벨 데이터를 진단합니다(왜 척도가 이렇게 다른가).
3. 4단계 '전체기준'은 미래 쇼트까지 기준에 넣어 현장에서 못 씁니다. **채점할 쇼트보다 과거의 쇼트만**으로 기준선을 세웁니다.

**외부 데이터** (보고서에 사유·방법·출처 명기)
- `moldset_labeled.csv` : 공식 라벨 파일의 원값 (`Unnamed: 0` 으로 1:1 연결)
- `unlabeled_raw_cn7.csv`, `unlabeled_raw_rg3.csv` : `unlabeled_data.csv`에서 공식 비라벨 `Unnamed: 0` 행만 꺼낸 원값 + 시각

세 파일을 `data/raw/`에 두고 실행하세요. 질문 칸에는 **실행 전에 예상을 먼저** 적습니다.

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
from src.official import *
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

ML = pd.read_csv(RAW / "moldset_labeled.csv")            # [외부] 라벨 원값
ML["ts"] = pd.to_datetime(ML["TimeStamp"])
MLI = ML.set_index("Unnamed: 0")

DATA, URAW = {}, {}
for p in ["CN7", "RG3"]:
    lab, unl = load(p)
    lab["ts"] = pd.to_datetime(lab["order"].map(MLI["TimeStamp"]))   # [외부] 시각
    DATA[p] = (lab, unl)
    u = pd.read_csv(RAW / f"unlabeled_raw_{p.lower()}.csv")         # [외부] 비라벨 원값
    u["ts"] = pd.to_datetime(u["TimeStamp"])
    URAW[p] = u
    print(p, "라벨", lab.shape, "비라벨", unl.shape, "비라벨 원값", u.shape)

### 5-1. 표준화값과 원값의 관계
표준화는 `표준화값 = (원값 − 평균) / 표준편차` 입니다. 바꿔 쓰면 `표준화값 = a × 원값 + b` (a = 1/표준편차, b = −평균/표준편차).
그래서 변수마다 원값과 표준화값을 점으로 찍으면 **직선 위에 정확히** 놓여야 합니다. 직선에서 평균과 표준편차를 거꾸로 구할 수 있습니다.

> **질문 5-1.** 직선 오차는 0일까요? 라벨 파일과 비라벨 파일의 평균·표준편차는 비슷할까요?
>
> 내 예상:

In [ ]:
def fit_scale(raw, std):
    """표준화값 = a*원값 + b 를 맞추고 (평균, 표준편차, 최대 오차)를 돌려준다."""
    if np.std(raw) == 0:
        return raw.mean(), 0.0, np.nan          # 값이 하나뿐인 변수
    a, b = np.polyfit(raw, std, 1)
    return -b / a, 1 / a, np.abs(a * raw + b - std).max()

rows = []
for p in ["CN7", "RG3"]:
    lab, unl = DATA[p]
    Lraw = MLI.loc[lab["order"], VARS].values
    u = URAW[p]
    assert (u["Unnamed: 0"].values == unl["order"].values).all()   # 같은 순서인지 확인
    for j, v in enumerate(VARS):
        lm, ls, le = fit_scale(Lraw[:, j], lab[v].values)
        um, us, ue = fit_scale(u[v].values, unl[v].values)
        rows.append({"제품": p, "변수": v, "라벨 평균": lm, "라벨 표준편차": ls, "라벨 오차": le,
                     "비라벨 평균": um, "비라벨 표준편차": us, "비라벨 오차": ue})
SCALE = pd.DataFrame(rows)
print("최대 직선 오차  라벨:", SCALE["라벨 오차"].max(), " 비라벨:", SCALE["비라벨 오차"].max())
SCALE.round(3)

### 5-2. 비라벨 진단
CN7 쿠션 위치는 라벨 파일에서 표준편차 0.005mm인데, 비라벨에서는 300mm가 넘습니다. 같은 설비, 같은 제품인데 왜일까요?

> **질문 5-2.** 비라벨 쿠션 위치 값을 히스토그램으로 그리면 어떤 모양일까요? (봉우리 하나 / 둘 / 넓게 퍼짐)
>
> 내 예상:

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, p in zip(axes, ["CN7", "RG3"]):
    ax.hist(URAW[p]["Cushion_Position"], bins=100)
    ax.set_title(f"{p} 비라벨 Cushion_Position 원값"); ax.set_yscale("log")
plt.show()

for p in ["CN7", "RG3"]:
    u = URAW[p]
    u["zero"] = u["Barrel_Temperature_1"] < 100      # 배럴 온도가 0으로 찍힌 행
    print(f"── {p}: 0 모드 {int(u['zero'].sum())}행 / 전체 {len(u)}행 ({u['zero'].mean():.0%})")
    print(pd.DataFrame({"정상 모드": u.loc[~u["zero"], VARS].median(),
                        "0 모드": u.loc[u["zero"], VARS].median()}).T.round(2).to_string(), "\n")

'0 모드' 행은 배럴·금형 온도, 스크루 RPM이 0이고 사이클이 약 36초입니다(정상은 약 60초). 사출 시간은 정확히 5.00초로 고정돼 있습니다.
설비 코드는 모두 우진2호기(S14)로 확인했습니다. 그렇다면 0 모드는 정상 생산 중간중간 끼어 있는 걸까요, 아니면 따로 떨어진 시간대일까요?

> **질문 5-3.** 0 모드 행 근처(±1분, ±2분, ±10분, ±1시간)에 정상 모드 행이 있는 비율은 얼마쯤일까요?
>
> 내 예상:

In [ ]:
for p in ["CN7", "RG3"]:
    u = URAW[p]
    zt = np.sort(u.loc[u["zero"], "ts"].values.astype("datetime64[s]").astype("int64"))
    nt = np.sort(u.loc[~u["zero"], "ts"].values.astype("datetime64[s]").astype("int64"))
    i = np.searchsorted(nt, zt)
    gap = np.minimum(np.abs(zt - nt[np.clip(i - 1, 0, len(nt) - 1)]), np.abs(nt[np.clip(i, 0, len(nt) - 1)] - zt))
    print(f"{p}: 0 모드 행 중 정상 행이 가까이 있는 비율  " +
          "  ".join(f"±{w // 60}분 {(gap <= w).mean():.0%}" for w in [60, 120, 600, 3600]))
    print(f"     사이클 간격 중앙값  0 모드 {np.median(np.diff(np.unique(zt))):.0f}초 / 정상 {np.median(np.diff(np.unique(nt))):.0f}초")

### 5-4. 라벨 쇼트는 비라벨 안에도 있을까?

In [ ]:
for p in ["CN7", "RG3"]:
    lab, _ = DATA[p]
    u = URAW[p]
    L = MLI.loc[lab["order"]].reset_index()
    m = u.merge(L[["ts", "PART_NAME"] + VARS], on=["ts", "PART_NAME"], suffixes=("", "_l"))
    same = np.isclose(m[VARS].values, m[[v + "_l" for v in VARS]].values, atol=1e-3).all(axis=1)
    print(f"{p}: 라벨 {len(lab)}행 중 비라벨에 시각·부품명·원값까지 같은 행 {int(same.sum())}")

### 5-5. 과거 기준선 이탈도 (현장형)
쇼트 i의 점수 = **쇼트 i보다 과거 시각**의 정상 모드 쇼트 K개를 기준으로 한 이탈도(4단계와 같은 robust z, rms). 값은 원래 단위.
- 같은 시각의 LH/RH는 한 쇼트로 셉니다.
- 라벨(PassOrFail)은 여전히 채점에만 씁니다.

두 가지 기준선을 비교합니다.
- **(가) 라벨 쇼트 제외**: 비라벨에서 라벨 쇼트를 빼고 과거 K쇼트 (불량이 기준선에 섞이지 않게)
- **(나) 과거 쇼트 전부**: 라벨 쇼트도 **값만** 포함 (현장에서는 검사 여부와 상관없이 과거 쇼트 값을 다 압니다)

> **질문 5-5.** (가)와 (나) 중 어느 쪽이 CN7에서 더 잘 될까요? (가)가 '더 깨끗한' 기준선인데 정말 그럴까요?
> 힌트: 라벨 쇼트는 CN7 10/16 04:57~08:05 하나의 런에 몰려 있습니다. 그 런 동안 (가)의 '과거 K쇼트'는 언제 쇼트일까요?
>
> 내 예상:

In [ ]:
def robust_z(x, P):
    med = np.median(P, axis=0)
    sd = np.maximum(np.median(np.abs(P - med), axis=0) * 1.4826, P.std(axis=0))
    ok = sd > 1e-6
    return np.where(ok, np.minimum(np.abs(x - med) / np.where(ok, sd, 1), 20), 0)


def past_baseline_score(lab, base, K):
    """라벨 쇼트마다, 그보다 과거 시각인 base 쇼트 K개로 기준을 세워 rms 이탈도. 행별 점수 반환."""
    sh = lab.drop_duplicates("pair")
    X = MLI.loc[sh["order"], VARS].values                      # 원값
    bt, BX = base["ts"].values, base[VARS].values
    sc = [np.sqrt((robust_z(x, BX[bt < t][-K:]) ** 2).mean()) for x, t in zip(X, sh["ts"].values)]
    return lab["pair"].map(pd.Series(sc, index=sh["pair"].values)).values


def both_rank(lab, s):
    """둘다불량 쇼트의 순위(위에서 %)."""
    sh = pd.Series(s, index=lab.index).groupby(lab["pair"]).first().rank(ascending=False, pct=True) * 100
    both = lab.groupby("pair")["y"].transform("sum") == 2
    return sh[lab.loc[both, "pair"].unique()].round(1).tolist()


rows = []
for p in ["CN7", "RG3"]:
    lab, _ = DATA[p]
    ceil = round(float(f1_ceiling(lab)["F1"]), 3)
    u = URAW[p]
    normal = u[~u["zero"]].drop_duplicates("ts").sort_values("ts")      # 정상 모드, 시각당 한 쇼트
    lab_ts = set(ML.loc[ML["PART_NAME"].str.startswith(p), "ts"])
    BASE = {"(가) 라벨 쇼트 제외": normal[~normal["ts"].isin(lab_ts)],
            "(나) 과거 쇼트 전부": normal}
    for name, base in BASE.items():
        for K in [10, 30, 100, 300]:
            s = past_baseline_score(lab, base, K)
            r = evaluate(lab, s, f"{name} K={K}", p, ceil)
            r["둘다불량 순위%"] = both_rank(lab, s)
            rows.append(r)
T5 = pd.DataFrame(rows)
T5[["모델", "제품", "AP", "무작위 평균 대비", "유의", "상위1%", "상위5%", "상위10%", "둘다불량 순위%"]].round(3)

> **질문 5-6.** (가)가 잘 안 된 이유를 한 줄로 적어 보세요. 그리고 (나)가 라벨을 쓰지 않는데도 '반칙'이 아닌 이유는?
>
> 내 답:

### 5단계 정리 (직접 적기)
1. 공식 라벨·비라벨 파일을 섞으려면 무엇이 필요했나? 비라벨에는 어떤 문제가 있었나?
2. 현장에서 쓸 수 있는(과거만 보는) 기준선으로도 4단계 결론이 유지되나?
3. RG3는?